# Bacteria movement: run and tumble
## session 1 of the course 

In [ ]:
# --------------------------
# Python libraries
# --------------------------
import sys
import numpy as np # numerical computing
import matplotlib.pyplot as plt # plotting
import pandas as pd # data handling
from pathlib import Path # file handling

import scipy.io as sio
import os


sys.path.append(os.path.abspath('../sandbox'))
import metrics # custom metrics for run and tumble analysis

import warnings
warnings.filterwarnings("ignore")

# the code below autoloads new changes (no restart of the kernel needed)
%reload_ext autoreload
%autoreload 2

In [ ]:
# --------------------------
# Notebook path
# --------------------------
os.getcwd() 

#### observation:
- In previous session, you learned how to load `.csv` data. here we have a `.mat` format which is MATLAB format
- instead we will use scipy.io to handle this MATLAB files (.mat). its best suitable for this

In [ ]:
# --------------------------
# loading the data
# --------------------------

base_path = os.path.join("./")
ecoli_path = os.path.join(base_path, "../data", "EcoliTrajectories.mat")

print("Base path:", base_path)
print("E. coli path:", ecoli_path)

ecoli_mat = sio.loadmat(ecoli_path)

print("Keys in E. coli file:", list(ecoli_mat.keys()))

# Looking into the data

In [ ]:
# --------------------------
# exploring the data structure
# --------------------------


all_keys = list(ecoli_mat.keys())
dataset_keys = [k for k in all_keys if k.startswith('V_')]

dataset_keys

In [ ]:
# --------------
# parameters
# -------------

angle_threshold = 45 # angle change threshold for run/tumble classification (degrees)
V_10min = ecoli_mat['V_10min'] # our dataset for now
ref_t = np.logspace(-1.5, 1.5, 100)
# The crossover between the two happens at τ_c (correlation time).
tau_c = 1.6  # τ_c ≈ 1.6 s for wild-type E. coli (from lecture).

In [ ]:
fps = float(V_10min['Parameters'][0, 0]['fps'][0, 0][0, 0])
dt  = 1.0 / fps
n_traj = len(V_10min['Speeds'][0, 0])
n_traj

In [ ]:
lengths = [len(V_10min['Speeds'][0,0][i][0]) for i in range(n_traj)]

plt.hist(lengths, bins=20, edgecolor='black')
plt.title("Distribution of Trajectory Lengths/durations")
plt.xlabel("Number of frames")
plt.ylabel("Count")
plt.show()

In [ ]:
raw = V_10min['Speeds'][0, 0][0][0]

for col in range(raw.shape[1]):
    n_nan = np.sum(np.isnan(raw[:, col]))
    print(f"    col {col}: {n_nan} NaN frames")

In [ ]:
# clean: keep only rows where ALL key columns are finite
key_cols  = [1, 2, 3, 5, 6, 7, 8, 9]   # x, y, z, speed, angle
valid     = np.all(np.isfinite(raw[:, key_cols]), axis=1)
n_dropped = (~valid).sum()
raw_clean = raw[valid]

In [ ]:
raw.shape[0], raw_clean.shape[0]

In [ ]:
df = pd.DataFrame({
    'frame'           : raw_clean[:, 0].astype(int),
    'time_s'          : raw_clean[:, 0] / fps,
    'x_um'            : raw_clean[:, 1],
    'y_um'            : raw_clean[:, 2],
    'z_um'            : raw_clean[:, 3],
    'vx_um_s'         : raw_clean[:, 5],  
    'vy_um_s'         : raw_clean[:, 6],  
    'vz_um_s'         : raw_clean[:, 7],  
    'speed_um_s'      : raw_clean[:, 8],  
    'angle_change_deg': raw_clean[:, 9],  
}).reset_index(drop=True)


In [ ]:
df